<span style="color:#8E44AD;">PCOS Prediction Model</span>

<span style="color:#BA68C8;">Dataset Loading</span>

In [13]:
import pandas as pd

df = pd.read_csv('../data/PCOS_data.csv')
df.head()
df.info()
df['PCOS (Y/N)'].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 45 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Sl. No                  541 non-null    int64  
 1   Patient File No.        541 non-null    int64  
 2   PCOS (Y/N)              541 non-null    int64  
 3    Age (yrs)              541 non-null    int64  
 4   Weight (Kg)             541 non-null    float64
 5   Height(Cm)              541 non-null    float64
 6   BMI                     541 non-null    float64
 7   Blood Group             541 non-null    int64  
 8   Pulse rate(bpm)         541 non-null    int64  
 9   RR (breaths/min)        541 non-null    int64  
 10  Hb(g/dl)                541 non-null    float64
 11  Cycle(R/I)              541 non-null    int64  
 12  Cycle length(days)      541 non-null    int64  
 13  Marraige Status (Yrs)   540 non-null    float64
 14  Pregnant(Y/N)           541 non-null    int64  
 15  

PCOS (Y/N)
0    364
1    177
Name: count, dtype: int64

<span style="color:#BA68C8;">Data Cleaning</span>

In [14]:
df = df.drop(columns=['Unnamed: 44'])

df.columns = df.columns.str.strip()

df['II    beta-HCG(mIU/mL)'.strip()] = pd.to_numeric(df['II    beta-HCG(mIU/mL)'.strip()], errors='coerce')
df['AMH(ng/mL)'] = pd.to_numeric(df['AMH(ng/mL)'], errors='coerce')

df.isnull().sum().sort_values(ascending=False).head(10)

Fast food (Y/N)           1
AMH(ng/mL)                1
II    beta-HCG(mIU/mL)    1
Marraige Status (Yrs)     1
Weight (Kg)               0
Height(Cm)                0
BMI                       0
Blood Group               0
Sl. No                    0
Patient File No.          0
dtype: int64

<span style="color:#BA68C8;">Missing Value Treatment</span>

In [15]:
for col in ['Marraige Status (Yrs)', 'Fast food (Y/N)', 'II    beta-HCG(mIU/mL)', 'AMH(ng/mL)']:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

print(df.isnull().sum().sum())

0


<span style="color:#BA68C8;">Remove Unnecessary Features</span>

In [16]:
df = df.drop(columns=['Sl. No', 'Patient File No.'])

correlations = df.corr(numeric_only=True)['PCOS (Y/N)'].sort_values(ascending=False)
print(correlations)

PCOS (Y/N)                1.000000
Follicle No. (R)          0.648327
Follicle No. (L)          0.603346
Skin darkening (Y/N)      0.475733
hair growth(Y/N)          0.464667
Weight gain(Y/N)          0.441047
Cycle(R/I)                0.401644
Fast food (Y/N)           0.376183
Pimples(Y/N)              0.286077
AMH(ng/mL)                0.264141
Weight (Kg)               0.211938
BMI                       0.199697
Hair loss(Y/N)            0.172879
Waist(inch)               0.164598
Hip(inch)                 0.162297
Avg. F size (L) (mm)      0.132992
Endometrium (mm)          0.106648
Avg. F size (R) (mm)      0.097690
Pulse rate(bpm)           0.091821
Hb(g/dl)                  0.087170
Vit D3 (ng/mL)            0.085494
Height(Cm)                0.068254
Reg.Exercise(Y/N)         0.065337
LH(mIU/mL)                0.063879
RBS(mg/dl)                0.048922
BP _Diastolic (mmHg)      0.038032
RR (breaths/min)          0.036928
Blood Group               0.036433
II    beta-HCG(mIU/m

<span style="color:#BA68C8;">Feature Engineering</span>

In [17]:

df['Cycle(R/I)'] = df['Cycle(R/I)'].replace({2: 0, 4: 1, 5: 1})



final_features = [
    'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'BMI',
    'Cycle(R/I)', 'Cycle length(days)', 'Pregnant(Y/N)',
    'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)',
    'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)'
]

X = df[final_features]
y = df['PCOS (Y/N)']

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)
X.head()

Feature matrix shape: (541, 14)
Target shape: (541,)


,Age (yrs),Weight (Kg),Height(Cm),BMI,Cycle(R/I),Cycle length(days),Pregnant(Y/N),Weight gain(Y/N),hair growth(Y/N),Skin darkening (Y/N),Hair loss(Y/N),Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N)
0,28,44.6,152.0,19.3,0,5,0,0,0,0,0,0,1.0,0
1,36,65.0,161.5,24.9,0,5,1,0,0,0,0,0,0.0,0
2,33,68.8,165.0,25.3,0,5,1,0,0,0,1,1,1.0,0
3,37,65.0,148.0,29.7,0,5,0,0,0,0,0,0,0.0,0
4,25,52.0,161.0,20.1,0,5,1,0,0,0,1,0,0.0,0


<span style="color:#BA68C8;">Train-Test Split</span>

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training set:', X_train.shape)
print('Testing set:', X_test.shape)

Training set: (432, 14)
Testing set: (109, 14)


<span style="color:#BA68C8;">Feature Scaling</span>

In [19]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaling done. Example row before:', X_train.iloc[0].values[:4])
print('Example row after:', X_train_scaled[0][:4])

Scaling done. Example row before: [ 25.   53.5 152.   23.2]
Example row after: [-1.18219286 -0.52990914 -0.72122274 -0.25584677]


<span style="color:#BA68C8;">Logistic Regression Model</span>

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)

print('Accuracy: ', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall:   ', recall_score(y_test, y_pred))
print('F1 Score: ', f1_score(y_test, y_pred))
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy:  0.8715596330275229
Precision: 0.84375
Recall:    0.75
F1 Score:  0.7941176470588235

Confusion Matrix:
[[68  5]
 [ 9 27]]


<span style="color:#BA68C8;">Train Additional Machine Learning Models</span>

In [21]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier()
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    }

results['Logistic Regression'] = {
    'Accuracy': accuracy_score(y_test, y_pred := log_reg.predict(X_test_scaled)),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred)
}

results_df = pd.DataFrame(results).T.sort_values('F1 Score', ascending=False)
results_df

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.871560,0.843750,0.750000,0.794118
SVM,0.853211,0.833333,0.694444,0.757576
Random Forest,0.844037,0.806452,0.694444,0.746269
KNN,0.816514,0.750000,0.666667,0.705882
Decision Tree,0.770642,0.657143,0.638889,0.647887


<span style="color:#BA68C8;">XGBoost Model</span>

In [22]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train)
y_pred = xgb_model.predict(X_test_scaled)

results['XGBoost'] = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred)
}

results_df = pd.DataFrame(results).T.sort_values('F1 Score', ascending=False)
results_df

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.871560,0.843750,0.750000,0.794118
SVM,0.853211,0.833333,0.694444,0.757576
Random Forest,0.844037,0.806452,0.694444,0.746269
KNN,0.816514,0.750000,0.666667,0.705882
XGBoost,0.816514,0.750000,0.666667,0.705882
Decision Tree,0.770642,0.657143,0.638889,0.647887


<span style="color:#BA68C8;">Save the Trained Model</span>

In [23]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(log_reg, '../models/pcos_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(final_features, '../models/feature_names.pkl')

print('Model, scaler, and feature list saved successfully.')

Model, scaler, and feature list saved successfully.
